# NB1 — Hooks & the Activation Cache

In NB0 we treated the model as a black box: `tokens → logits`. This is where that changes.

Recall the picture from our last discussion: a **residual stream** of shape `[batch, pos, d_model]`
flows through 12 blocks, and each block *reads* it and *adds* its contribution back. TransformerLens
exposes **every one of those intermediate tensors** through named **hook points**. This notebook is
about grabbing them.

You'll learn:

1. `run_with_cache` — run the model and capture *all* activations in one shot.
2. The **hook naming scheme** — how every activation gets a string name like
   `blocks.0.attn.hook_pattern`.
3. `utils.get_act_name` and the `cache["name", layer]` **shortcut** for indexing without typing full
   strings.
4. The **residual stream** in the cache (`resid_pre` / `resid_mid` / `resid_post`) and the additive
   read/write view.
5. **Attention patterns** — a `[batch, head, dest, src]` tensor — which we'll hunt through in NB2/NB3.

## 0. Setup

Same as NB0: gradients off, model on `cuda`.

In [1]:
import torch
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = HookedTransformer.from_pretrained("gpt2", device=device)
print("loaded:", model.cfg.model_name, "| device:", device)

C:\Users\alexa\AppData\Local\Temp\ipykernel_39752\1778070597.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
loaded: gpt2 | device: cuda


## 1. `run_with_cache`

`model.run_with_cache(tokens)` does the same forward pass as `model(tokens)` but **also** returns an
`ActivationCache` — a dict-like object holding every intermediate activation the model produced.

So you get *two* things back: the usual `logits`, and the `cache`.

In [2]:
text = "The cat sat on the mat"
tokens = model.to_tokens(text)

logits, cache = model.run_with_cache(tokens)

print("logits shape:", logits.shape)          # [1, seq, d_vocab] — same as NB0
print("cache type:", type(cache).__name__)
print("number of cached activations:", len(cache))

logits shape: torch.Size([1, 7, 50257])
cache type: ActivationCache
number of cached activations: 210


## 2. The hook naming scheme

Every cached activation has a **string name**. The names are structured and predictable:

```
blocks.{layer}.{submodule}.{hook}
```

for example:

- `blocks.0.hook_resid_pre`   — residual stream entering block 0
- `blocks.0.attn.hook_pattern` — attention pattern (post-softmax) in block 0
- `blocks.3.hook_mlp_out`      — what block 3's MLP writes back into the stream

Plus a few non-block ones like `hook_embed` (token embeddings) and `ln_final.hook_normalized`.

Let's look at the names for **block 0** only, so the list is readable.

In [3]:
# Every key in the cache is one of these structured names.
for name in cache.keys():
    if name.startswith("blocks.0.") or not name.startswith("blocks."):
        print(f"{name:35}  {tuple(cache[name].shape)}")

hook_embed                           (1, 7, 768)
hook_pos_embed                       (1, 7, 768)
blocks.0.hook_resid_pre              (1, 7, 768)
blocks.0.ln1.hook_scale              (1, 7, 1)
blocks.0.ln1.hook_normalized         (1, 7, 768)
blocks.0.attn.hook_q                 (1, 7, 12, 64)
blocks.0.attn.hook_k                 (1, 7, 12, 64)
blocks.0.attn.hook_v                 (1, 7, 12, 64)
blocks.0.attn.hook_attn_scores       (1, 12, 7, 7)
blocks.0.attn.hook_pattern           (1, 12, 7, 7)
blocks.0.attn.hook_z                 (1, 7, 12, 64)
blocks.0.hook_attn_out               (1, 7, 768)
blocks.0.hook_resid_mid              (1, 7, 768)
blocks.0.ln2.hook_scale              (1, 7, 1)
blocks.0.ln2.hook_normalized         (1, 7, 768)
blocks.0.mlp.hook_pre                (1, 7, 3072)
blocks.0.mlp.hook_post               (1, 7, 3072)
blocks.0.hook_mlp_out                (1, 7, 768)
blocks.0.hook_resid_post             (1, 7, 768)
ln_final.hook_scale                  (1, 7, 1)
ln_final

## 3. `utils.get_act_name` and the cache shortcut

Typing `"blocks.0.attn.hook_pattern"` by hand is error-prone. Two conveniences:

- **`utils.get_act_name(short, layer)`** builds the full string for you.
  `utils.get_act_name("pattern", 0)` → `"blocks.0.attn.hook_pattern"`.
- **`cache[short, layer]`** does the same lookup inline — `cache["pattern", 0]` is exactly
  `cache["blocks.0.attn.hook_pattern"]`.

These three lines all fetch the identical tensor:

In [ ]:
name = utils.get_act_name("pattern", 0)
print("get_act_name('pattern', 0) ->", name)

a = cache["blocks.0.attn.hook_pattern"]   # full string
b = cache[name]                            # built by get_act_name
c = cache["pattern", 0]                    # the tuple shortcut

print("all three identical:", torch.equal(a, b) and torch.equal(b, c))
print("shape:", tuple(c.shape))            # [batch, n_heads, dest_pos, src_pos]

get_act_name('pattern', 0) -> blocks.0.attn.hook_pattern
all three identical: True
shape: (1, 12, 7, 7)


## 4. The residual stream (the additive view)

This is the heart of it. Three residual-stream hooks bracket each block:

- `resid_pre`  — the stream **entering** the block.
- `resid_mid`  — after attention has been added, **before** the MLP.
- `resid_post` — after the MLP has been added; this is what feeds the next block.

All three have shape `[batch, pos, d_model]`. And remember the key claim from last time — each block
*adds* to the stream rather than overwriting it:

```
resid_mid  = resid_pre + attn_out
resid_post = resid_mid + mlp_out
```

Let's *prove* that from the cache, not just assert it.

In [5]:
layer = 0
resid_pre  = cache["resid_pre", layer]
resid_mid  = cache["resid_mid", layer]
resid_post = cache["resid_post", layer]
attn_out   = cache["attn_out", layer]    # what this block's attention writes to the stream
mlp_out    = cache["mlp_out", layer]     # what this block's MLP writes to the stream

print("resid_pre shape:", tuple(resid_pre.shape))   # [batch, pos, d_model]

# The stream is purely additive — verify to floating-point tolerance:
print("resid_mid  == resid_pre + attn_out :", torch.allclose(resid_mid, resid_pre + attn_out, atol=1e-4))
print("resid_post == resid_mid + mlp_out  :", torch.allclose(resid_post, resid_mid + mlp_out, atol=1e-4))

resid_pre shape: (1, 7, 768)
resid_mid  == resid_pre + attn_out : True
resid_post == resid_mid + mlp_out  : True


## 5. Attention patterns

The attention pattern is the tensor we'll obsess over for the rest of the course. Shape:

```
cache["pattern", layer]   ->   [batch, n_heads, dest_pos, src_pos]
```

Read it as: for each head, `pattern[head, dest, src]` is **how much the token at `dest` attends to
the token at `src`**. Two properties fall straight out of how attention is defined:

- It's **post-softmax**, so for a fixed `dest`, the weights over `src` **sum to 1**.
- It's **causal**: a token can only attend to itself and earlier tokens, so
  `pattern[head, dest, src] == 0` whenever `src > dest`. The matrix is lower-triangular.

Let's confirm both for one head.

In [6]:
pattern = cache["pattern", 0]          # [batch, n_heads, dest, src]
print("pattern shape:", tuple(pattern.shape))

head0 = pattern[0, 0]                   # batch 0, head 0  ->  [dest, src]
print("one head shape:", tuple(head0.shape))

# Each destination row is a probability distribution over source positions:
print("row sums (should all be ~1):", head0.sum(dim=-1))

# Causal: everything strictly above the diagonal is zero.
upper = torch.triu(head0, diagonal=1)
print("max weight above diagonal (should be 0):", upper.max().item())

pattern shape: (1, 12, 7, 7)
one head shape: (7, 7)
row sums (should all be ~1): tensor([1., 1., 1., 1., 1., 1., 1.], device='cuda:0')
max weight above diagonal (should be 0): 0.0


## Practice

Your turn. Fill in the `TODO(you)` cell to **extract one specific head's attention pattern by hand**
and sanity-check it.

Target: **layer 5, head 1** (write it as `LAYER, HEAD = 5, 1`).

Do all of the following:
1. Get that head's pattern **two different ways** and confirm they're identical:
   - (a) via the full hook-name string (build it with `utils.get_act_name`),
   - (b) via the `cache["pattern", LAYER]` shortcut.
   Both should give a `[dest, src]` matrix once you index batch 0 and the head.
2. Verify every **destination row sums to 1** (allow tolerance — use `torch.allclose` against a
   tensor of ones).
3. Verify the pattern is **causal** (nothing above the diagonal).

Hints: the head-1 slice is `[..., HEAD, :, :]` after picking batch 0. `torch.ones_like(row_sums)`
is handy for the tolerance check. `torch.triu(m, diagonal=1)` grabs the strict upper triangle.

In [8]:
LAYER, HEAD = 5, 1

name = utils.get_act_name("pattern", LAYER)
pattern1 = cache[name][0,HEAD]
pattern2 = cache["pattern", LAYER][0,HEAD]
print(torch.equal(pattern1,pattern2))
row_sums = pattern1.sum(dim=-1)
print(row_sums)
upper = torch.triu(pattern1, diagonal=1)
print("max weight above diagonal (should be 0):", upper.max().item())

# TODO(you):
# 1) name = ...                     # full hook name via utils.get_act_name
#    pattern_a = cache[name][0, HEAD]        # [dest, src]
#    pattern_b = cache["pattern", LAYER][0, HEAD]
#    assert torch.equal(pattern_a, pattern_b)
#
# 2) row_sums = ...                 # sum over src
#    print("rows sum to 1:", ...)
#
# 3) print("causal:", ...)          # max above diagonal == 0


True
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       device='cuda:0')
max weight above diagonal (should be 0): 0.0


<details>
<summary>Reference solution (open after you've tried)</summary>

```python
LAYER, HEAD = 5, 1

# 1) two ways, must match
name = utils.get_act_name("pattern", LAYER)      # 'blocks.5.attn.hook_pattern'
pattern_a = cache[name][0, HEAD]                  # [dest, src]
pattern_b = cache["pattern", LAYER][0, HEAD]
assert torch.equal(pattern_a, pattern_b)
print("both routes identical, shape:", tuple(pattern_a.shape))

# 2) rows are probability distributions
row_sums = pattern_a.sum(dim=-1)
print("rows sum to 1:", torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-4))

# 3) causal
print("causal:", torch.triu(pattern_a, diagonal=1).max().item() == 0.0)
```

</details>

---
**Done?** Once your practice cell confirms the two routes match, rows sum to 1, and the pattern is
causal, ask me to review. Next up is **NB2 — attention visualization**, where instead of staring at
numbers we render these `[dest, src]` matrices with `circuitsvis` and start *reading* what heads do.